# Extractive retrieval pipeline

This notebook demonstrates the full retrieval pipeline: you ask a question, the system finds and ranks the most relevant passages from your documents, and returns them with sources. The key point is that **no text is generated** — everything you see comes verbatim from the corpus.

Run notebook 02 first to populate the ChromaDB collection.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.embeddings.generator import EmbeddingGenerator
from src.storage.chroma_store import ChromaVectorStore
from src.retrieval.pipeline import RetrievalPipeline

gen = EmbeddingGenerator('fast')
store = ChromaVectorStore(persist_dir='../chroma_data', collection_name='notebook_demo')
pipeline = RetrievalPipeline(store, gen)

In [ ]:
questions = [
    'What is self-attention and why does it matter?',
    'How does BM25 keyword search work?',
    'Why is cosine similarity used for text embeddings?',
    'What are the differences between ChromaDB and pgvector?',
]

for q in questions:
    result = pipeline.query(q, n=2)
    print(f'Q: {q}')
    print(f'Sources: {result["sources"]}')
    print(f'Best snippet: {result["best_snippet"][:300]}…')
    print()

In [ ]:
# Inspect a full result object
result = pipeline.query('what is positional encoding?', n=3)

print('Full result structure:\n')
for i, p in enumerate(result['passages']):
    print(f"  Passage {i+1} (score {p['score']:.3f}) — {p['metadata'].get('source', '?')}")
    print(f"  {p['text'][:250]}\n")

In [ ]:
# UMAP cluster visualisation
from src.clustering.umap_reducer import UMAPReducer
from src.clustering.kmeans_cluster import KMeansCluster
import plotly.express as px
import pandas as pd

reducer = UMAPReducer()
points = reducer.reduce(store, collection='notebook_demo')

if points:
    clusterer = KMeansCluster(n_clusters=3)
    cluster_labels = clusterer.cluster(store, collection='notebook_demo')
    label_map = {c['id']: str(c['cluster']) for c in cluster_labels}
    for p in points:
        p['cluster'] = label_map.get(p['id'], '0')

    df = pd.DataFrame(points)
    fig = px.scatter(
        df, x='x', y='y', color='cluster',
        hover_data=['text'], title='UMAP — document chunks',
        width=800, height=500
    )
    fig.show()
else:
    print('Not enough data for UMAP — run notebook 02 first.')